# Dataset

## Retrieving data

In [1]:
from src.datasets import save_pages_to_jsonl

save_pages_to_jsonl(
    dataset_name="FineWeb Edu 2",
    num_pages=1000,
    max_workers = 6,
    output_path="./data/fineweb_edu_2/fineweb_edu_2-train.jsonl",
    random_seed=163,
)

Saving 1000 pages from FineWeb Edu 2 to ./data/fineweb_edu_2/fineweb_edu_2-train.jsonl
Fetching dataset configs...
Fetching pages...
Writing 50000 pages to ./data/fineweb_edu_2/fineweb_edu_2-train.jsonl...


Writing to file: 100%|██████████| 50000/50000 [00:02<00:00, 19584.40it/s]


Done!


In [2]:

save_pages_to_jsonl(
    dataset_name="FineWeb Edu 2",
    num_pages=1000,
    output_path="./data/fineweb_edu_2/fineweb_edu_2-validation.jsonl",
    max_workers = 10,
    random_seed=164,
)


Saving 1000 pages from FineWeb Edu 2 to ./data/fineweb_edu_2/fineweb_edu_2-validation.jsonl
Fetching dataset configs...
Fetching pages...
Writing 50000 pages to ./data/fineweb_edu_2/fineweb_edu_2-validation.jsonl...


Writing to file: 100%|██████████| 50000/50000 [00:02<00:00, 20359.46it/s]


Done!


In [3]:

save_pages_to_jsonl(
    dataset_name="FineWeb Edu 2",
    num_pages=1000,
    output_path="./data/fineweb_edu_2/fineweb_edu_2-test.jsonl",
    max_workers = 4,
    random_seed=165,
)

Saving 1000 pages from FineWeb Edu 2 to ./data/fineweb_edu_2/fineweb_edu_2-test.jsonl
Fetching dataset configs...
Fetching pages...
Failed to fetch data, retrying with a newly sampled page. Attempt 1/10000
Writing 50000 pages to ./data/fineweb_edu_2/fineweb_edu_2-test.jsonl...


Writing to file: 100%|██████████| 50000/50000 [00:02<00:00, 19651.41it/s]


Done!


## Uploading dataset to huggingface

In [ ]:
from huggingface_hub import HfApi, login
import os
from dotenv import load_dotenv

# Load token from .sky_env file
load_dotenv(".sky_env")
token = os.getenv("HF_TOKEN")

# Login to Hugging Face
if token is None:
    login()  # This will prompt for token interactively
else:
    login(token=token)  # Login with token from .env

api = HfApi(token=token)

# Create the dataset repository
api.create_repo(
    repo_id="deepcoreCoalbiter/fineweb_edu_2", repo_type="dataset", private=False
)

# Upload the dataset files
api.upload_folder(
    repo_id="deepcoreCoalbiter/fineweb_edu_2",
    folder_path="./data/fineweb_edu_2",
    repo_type="dataset",
)

# Single Merge

In [1]:
import yaml

MODEL_NAME = "test3"
yaml_config = """
slices:
  - sources:
      - model: RoyJoy/llama-jan08
        layer_range: [0, 48]
      - model: RoyJoy/llama-jan16
        layer_range: [0, 48]
merge_method: slerp
base_model: RoyJoy/llama-jan08
parameters:
  t:
    - filter: self_attn
      value: [0, 0.5, 0.3, 0.7, 1]
    - filter: mlp
      value: [1, 0.5, 0.7, 0.3, 0]
    - value: 0.5
dtype: bfloat16
"""

# Save config as yaml file
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [ ]:
# run merge
!mergekit-yaml config.yaml merge --lazy-unpickle

In [ ]:
!pip install -qU huggingface_hub
!pip install python-dotenv

from huggingface_hub import ModelCard, ModelCardData
from jinja2 import Template

username = "deepcoreCoalbiter"

template_text = """
# {{ model_name }}
"""

# Create a Jinja template object
jinja_template = Template(template_text.strip())

# Get list of models from config
data = yaml.safe_load(yaml_config)
if "models" in data:
    models = [data["models"][i]["model"] for i in range(len(data["models"])) if "parameters" in data["models"][i]]
elif "parameters" in data:
    models = [data["slices"][0]["sources"][i]["model"] for i in range(len(data["slices"][0]["sources"]))]
elif "slices" in data:
    models = [data["slices"][i]["sources"][0]["model"] for i in range(len(data["slices"]))]
else:
    raise Exception("No models or slices found in yaml config")

# Fill the template
content = jinja_template.render(
    model_name=MODEL_NAME,
)

# Save the model card
card = ModelCard(content)
card.save('merge/README.md')


In [ ]:
from huggingface_hub import HfApi

username = "deepcoreCoalbiter"

# Read token from .env file
import os
from dotenv import load_dotenv

load_dotenv()

api = HfApi(token=os.getenv("HF_TOKEN"))

api.create_repo(repo_id=f"{username}/{MODEL_NAME}", repo_type="model")
api.upload_folder(
    repo_id=f"{username}/{MODEL_NAME}",
    folder_path="merge",
)

# Evolutionary merge with custom task

In [5]:
yaml_config = """
genome:
    models:
      - yfarm01/sn29_jan23_c0
      - RoyJoy/llama-jan16
      - luaqi/llama_01141
      - Rich-J/subnet29_upload_c00_Jan17_0
      - mci29/sn29_x2m5_dsne
      - RoyJoy/llama-jan08
    merge_method: dare_ties
    base_model: yfarm01/sn29_jan23_c0
    layer_granularity: 2

tasks:
  - name: fine_web_edu_2_ppl
    weight: 1.0
    metric: word_perplexity
"""

# Save config as yaml file
with open("evolve_config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [ ]:
# !mergekit-evolve -vllm --strategy pool --task-search-path lm-eval-tasks/fineweb-edu-2 --storage-path ./evolve_storage evolve_config.yaml

In [ ]:
MODEL_NAME = "timmy"
from huggingface_hub import HfApi

username = "deepcoreCoalbiter"

# Read token from .env file
import os
from dotenv import load_dotenv

load_dotenv(".sky_env")

api = HfApi(token=os.getenv("HF_TOKEN"))

api.create_repo(repo_id=f"{username}/{MODEL_NAME}", repo_type="model")
api.upload_folder(
    repo_id=f"{username}/{MODEL_NAME}",
    folder_path="evolve_storage/final_model",
)